## Assignments

In both tasks:
- extend the code in the repositories src folder for the implementation part a)
- write the code in jupyter for part b)

<b>8/2 Improve delta calculation for FlatVol model <b>

In BS model the spot at time t is $ S_t = S_0 e^{(r-\frac{\sigma^2}{2})t+\sigma W_t} $. Knowing this, when we bump the spot we dont have to resimulate the spot, instead we can obtain it by scaling the inital simulation: $ S^{bumped}_t = S_0*(1+\delta) e^{(r-\frac{\sigma^2}{2})t+\sigma W_t} $.

a, implement this delta calculation method for MonteCarlo pricer (10p)

b, compare the value and the calculation time of the improved delta with the default bump and revaluation delta. (5p)


#### Solution

For part (a) I used the help of ChatGPT.

No lets compare the 2 methods. For the basic delta calculation, we did the MC simulation once and then bumped the price, then calculated the fair value again with the MC simulation. For the new method, we only do the MC simulation once and we can just multpiply the paths. So we expect the new method to be approximately 2 times faster.

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

current = Path().resolve()
sys.path.append(str(current))
sys.path.append(str(current.parents[2]))

from src.enums import *
from src.utils import *
from src.market_data import *
from src.pricer import *
# # Make charts interactive
# %matplotlib notebook

# Initialize market data
MarketData.initialize()

QuantCourseBP 439d6a142f71633bcedd46f2e2a0e8d449967753*


Here, tenor_frequency plays a big part. If its 0 (it can be for european call), the the run time is basically the same because we only simulate the terminal prices, so simulation itself is cheap and the runtime is dominated by Python overhead.

In [2]:
import time

und = Stock.TEST_COMPANY
model = BSVolModel(und)

contract = EuropeanContract(
    und,
    PutCallFwd.CALL,
    LongShort.LONG,
    100,
    1
)

params_MC = MCParams(seed=1, num_of_path=1000, tenor_frequency=0)

pricer_MC = GenericMCPricer(contract, model, params_MC)

start = time.time()
delta_improved = pricer_MC.calc_delta(GreekMethod.BUMP)
time_improved = time.time() - start

start = time.time()
delta_default = Pricer.calc_delta(pricer_MC, GreekMethod.BUMP)
time_default = time.time() - start

print("Improved MC delta:", delta_improved)
print("Improved MC time:", time_improved)

print("Default bump-and-revalue delta:", delta_default)
print("Default bump-and-revalue time:", time_default)

Improved MC delta: 0.6732903264700925
Improved MC time: 0.031055212020874023
Default bump-and-revalue delta: 0.6732903264700898
Default bump-and-revalue time: 0.04932808876037598


However, if we have larger frequency values (For example, if we need extra monitoring points other than the default ones implemented for asian and barrier), the run time gets larger. For tenor_frequency = 252, we can see the expected 2x difference. The runtime is 2 times as large for the default delta method, but we get the same delta value for both cases.

In [3]:
params_MC = MCParams(seed=1, num_of_path=1000, tenor_frequency=252)

pricer_MC = GenericMCPricer(contract, model, params_MC)

start = time.time()
delta_improved = pricer_MC.calc_delta(GreekMethod.BUMP)
time_improved = time.time() - start

start = time.time()
delta_default = Pricer.calc_delta(pricer_MC, GreekMethod.BUMP)
time_default = time.time() - start

print("Improved MC delta:", delta_improved)
print("Improved MC time:", time_improved)

print("Default bump-and-revalue delta:", delta_default)
print("Default bump-and-revalue time:", time_default)

Improved MC delta: 0.6270869706486568
Improved MC time: 4.706879377365112
Default bump-and-revalue delta: 0.627086970648655
Default bump-and-revalue time: 9.387865781784058
